# Stage 4: proxies

**Goal.** Give every training patch and every test tile its landscape description, then express
each test tile as a **position in the training distribution**. That position is the tile's
per-proxy out-of-distribution score.

**Command-line equivalent:** `p037 proxies --config <config>`

**Code:** `src/p037/proxies.py` (coverage), `src/p037/ood.py` (placement). The proxy values
themselves are computed by `analysis/proxy/compute_proxies.py` (PR #3).

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from p037 import plots                      # sets the figure style used in the report
from p037.config import REPO_ROOT, load_config
from p037.datasets import get_dataset

# The experiment this notebook works on. smoke.yaml is generated data: it runs anywhere in minutes
# and its numbers mean nothing. On BluePebble use analysis/p037/configs/exp_b_usf.yaml.
# (Setting the environment variable P037_CONFIG before starting Jupyter does the same for all notebooks.)
CONFIG = os.environ.get("P037_CONFIG", "analysis/p037/configs/smoke.yaml")
cfg = load_config(REPO_ROOT / CONFIG)
ds = get_dataset(cfg)
DRY_RUN = cfg.dataset == "synthetic"
print(f"experiment {cfg.name!r} on dataset {cfg.dataset!r}")
print("dataset files:", cfg.dataset_dir)
print("run folder:   ", cfg.out)

## 4.1 Is every sample covered?

The proxy table has one row per location: training patches (the reference set) and test tiles.
`check_coverage` compares it with what this experiment needs. For anything missing it writes the
list with coordinates and prints the command that computes the values (that step downloads from
public servers, so it needs internet). For the dry run the values are simply made up.

In [ ]:
from p037.proxies import check_coverage

try:
    coverage = check_coverage(cfg)
    print(f"{coverage['covered']} of {coverage['needed']} samples have proxy values")
    display(coverage["by_set"])
except FileNotFoundError as error:      # e.g. the zero-shot config: no cache and no tiles.csv in its run folder
    print("coverage not checked:", error)

In [ ]:
proxy = pd.read_csv(cfg.resolve(cfg.proxy_table))
columns = {p: f"{p}__{cfg.proxy_scale}" for p in cfg.proxies}      # proxy name -> its column at the chosen scale
ref = proxy[proxy["set"] == cfg.reference_set]                      # the training distribution
test = proxy[proxy["set"] == "test"]
print(f"reference set {cfg.reference_set!r}: {len(ref)} rows | test tiles: {len(test)} | scale: {cfg.proxy_scale}")
proxy[["sample_id", "set", *columns.values()]].groupby("set").describe().T.round(3).head(16)

The proxies, and what each is meant to capture:

| Proxy | Source | Why it could matter for a SAR flood model |
|---|---|---|
| `slope_mean`, `slope_p90`, `elev_range` | Copernicus DEM GLO-30 | radar shadow and layover in relief; water does not pool on slopes |
| `hand_median`, `hand_lt5_frac` | GLO-30 HAND (height above nearest drainage) | how much of the tile can flood at all |
| `veg_frac`, `tree_frac` | ESA WorldCover 2021 | vegetation hides water or changes its radar signature |
| `built_frac` | GHSL | buildings brighten flooded streets (double bounce) |

WSF2019 is deliberately not used: UrbanSARFloods defines its urban-flood labels with it.

## 4.2 Training distribution against each test scene

The raw comparison: for each proxy, where do a scene's tiles fall relative to what the model was
trained on? A scene whose histogram sits where the grey one is empty is out of distribution on
that proxy.

In [ ]:
events = sorted(test["event"].dropna().unique())
shown = list(columns)[:4]
fig, axes = plt.subplots(len(shown), len(events), figsize=(3 * len(events), 1.9 * len(shown)), squeeze=False, sharex="row")
for i, p in enumerate(shown):
    lo, hi = np.nanpercentile(proxy[columns[p]], [1, 99])
    bins = np.linspace(lo, hi if hi > lo else lo + 1, 30)
    for j, event in enumerate(events):
        ax = axes[i, j]
        ax.hist(ref[columns[p]].dropna(), bins=bins, density=True, color=plots.MUTED, alpha=0.55, label="training")
        ax.hist(test.loc[test["event"] == event, columns[p]].dropna(), bins=bins, density=True, color=plots.BLUE,
                alpha=0.6, label="test tiles")
        ax.set_yticks([])
        if i == 0:
            ax.set_title(plots.nice_event(event), loc="left")
        if j == 0:
            ax.set_ylabel(plots.nice_proxy(p), rotation=0, ha="right", va="center")
axes[0, 0].legend(fontsize=7)
fig.tight_layout()

## 4.3 From a raw value to a score

Three numbers per tile and proxy, all from the training distribution:

- **percentile** (0 to 100): the share of training samples with a lower value.
- **position** = (percentile - 50) / 50, from -1 to +1. Signed: "more built-up than training".
- **distance** = |position|, from 0 to 1. Unsigned: "unlike training, on either side".
  This is the out-of-distribution score proper.

Why not sigma (z-score), as first proposed? Several proxies are mostly zero with a long tail
(built-up share, tree cover in dry regions). Their training standard deviation is tiny, so a
z-score explodes for ordinary tiles. The percentile has no such problem. Both are computed, so
the two can be compared in the report.

In [ ]:
from p037 import ood

placed = ood.place(test.rename(columns={"sample_id": "tile_id"}).copy(), ref, columns)
p = list(columns)[0]
placed[["tile_id", "event", columns[p], f"pct__{p}", f"pos__{p}", f"dist__{p}", f"z__{p}"]].head(8).round(3)

In [ ]:
# share of each scene's tiles outside the central 90 % of the training data, per proxy
outside = pd.DataFrame({plots.nice_proxy(p): placed.groupby("event")[f"pct__{p}"].apply(
    lambda v: 100 * float(((v < 5) | (v > 95)).mean())) for p in columns}).round(0)
outside.index = [plots.nice_event(e) for e in outside.index]
print("percent of tiles outside the 5th to 95th percentile of the training data (10 % expected by chance)")
outside

## 4.4 Proxies that say the same thing

Mean slope, the 90th percentile of slope and elevation range all measure relief. Put together in
one regression, their weights would be unstable and uninterpretable. So before any regression,
a proxy whose rank correlation with one already kept exceeds `max_proxy_corr` (in the training
data) is dropped. The order of `proxies` in the config decides which of a pair is kept.

In [ ]:
kept, dropped = ood.prune_collinear(ref, columns, cfg.max_proxy_corr)
print("kept:   ", kept)
print("dropped:", {p: f"|rho| {abs(rho):.2f} with {twin}" for p, (twin, rho) in dropped.items()})

corr = ref[list(columns.values())].corr(method="spearman")
fig, ax = plt.subplots(figsize=(5.2, 4.4))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
names = [plots.nice_proxy(p) for p in columns]
ax.set_xticks(range(len(names)), names, rotation=45, ha="right"); ax.set_yticks(range(len(names)), names)
ax.grid(visible=False)
for i in range(len(names)):
    for j in range(len(names)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7,
                color="white" if abs(corr.iloc[i, j]) > 0.6 else plots.INK)
ax.set_title("Rank correlation between proxies, training data", loc="left")
fig.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout()

## Things to keep in mind when writing this up

- The reference set must be the data the model under test was **trained on**. For the pretrained
  Amazon model that is `a_amazon` (600 sampled points, an approximation); for Experiment B it is
  `b_usf_train`, narrowed automatically to the patches the run actually used.
- Proxies are measured in a fixed box of about 4.7 km around the tile centre (`proxy_scale: tile`)
  or about 24 km (`neigh`). Which scale is meaningful differs by proxy.
- The land cover and building layers date from 2018 to 2021, not the year of each flood.

Next: `05_analyze.ipynb`.